In [1]:
import os
import warnings
import numpy as np
import pandas as pd

from collections import Counter
from tqdm import tqdm

warnings.filterwarnings("ignore")

In [2]:
# ----------------------------
# Configuration
# ----------------------------

INPUT_FILE = "../dataset/processed/final_dataset.csv"

OUTPUT_DIR = "../dataset/engineered"

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "engineered_dataset.csv"
)

CHUNK_SIZE = 50000

WINDOW_SIZE = 100

STRIDE = 20

RENYI_ALPHA = 2

In [3]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output Folder:")
print(OUTPUT_DIR)

Output Folder:
../dataset/engineered


In [4]:
chunk_iterator = pd.read_csv(
    INPUT_FILE,
    chunksize=CHUNK_SIZE
)

first_chunk = next(chunk_iterator)

print(first_chunk.shape)

first_chunk.head()

(50000, 63)


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Inbound,Label
0,60675,80,6,5220876,12,9,1928.0,4290.0,509.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,1,DrDoS_NTP
1,60676,80,6,12644252,5,2,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,1,DrDoS_NTP
2,443,50458,6,3,2,0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,1,BENIGN
3,443,50465,6,3,2,0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,1,BENIGN
4,0,0,0,114329232,52,0,0.0,0.0,0.0,0.0,...,7.416667,2.466441,15.0,6.0,9527428.0,248706.681286,9950741.0,9092248.0,0,BENIGN


In [5]:
print(first_chunk.columns.tolist())

['Source Port', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'SYN Flag Count', 'RST Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Init_Win_bytes_forward', 'Init_Win

In [6]:
# Features used for statistical analysis

STAT_FEATURES = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Packet Length Mean",
    "Packet Length Std",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Average Packet Size"
]

print("Number of Statistical Features:", len(STAT_FEATURES))
print(STAT_FEATURES)

Number of Statistical Features: 8
['Flow Duration', 'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean', 'Packet Length Std', 'Total Fwd Packets', 'Total Backward Packets', 'Average Packet Size']


In [7]:
def calculate_renyi_entropy(values, alpha=2):
    """
    Calculate Rényi entropy of a numeric feature.
    """

    values = np.asarray(values)

    values = values[np.isfinite(values)]

    if len(values) == 0:
        return 0.0

    # Histogram
    hist, _ = np.histogram(values, bins=20)

    prob = hist / np.sum(hist)

    prob = prob[prob > 0]

    if len(prob) == 0:
        return 0.0

    if alpha == 1:
        return -np.sum(prob * np.log2(prob))

    entropy = (1 / (1 - alpha)) * np.log2(np.sum(prob ** alpha))

    return float(entropy)

In [8]:
def extract_statistics(window):

    feature_vector = {}

    for feature in STAT_FEATURES:

        values = window[feature].values

        feature_vector[f"{feature}_Mean"] = np.mean(values)

        feature_vector[f"{feature}_Std"] = np.std(values)

        feature_vector[f"{feature}_Variance"] = np.var(values)

        feature_vector[f"{feature}_Min"] = np.min(values)

        feature_vector[f"{feature}_Max"] = np.max(values)

        feature_vector[f"{feature}_Renyi"] = calculate_renyi_entropy(
            values,
            alpha=RENYI_ALPHA
        )

    return feature_vector

In [9]:
def create_sliding_windows(data,
                           window_size=WINDOW_SIZE,
                           stride=STRIDE):

    for start in range(
        0,
        len(data) - window_size + 1,
        stride
    ):

        end = start + window_size

        yield data.iloc[start:end]

In [10]:
def process_chunk(chunk):
    """
    Process a single chunk of flow records into engineered feature vectors.
    
    Parameters
    ----------
    chunk : pandas.DataFrame
        Input chunk of network flows.

    Returns
    -------
    pandas.DataFrame
        Engineered feature dataset for the chunk.
    """

    engineered_rows = []

    # Iterate through all sliding windows
    for window in create_sliding_windows(
        chunk,
        window_size=WINDOW_SIZE,
        stride=STRIDE
    ):

        # Extract statistical features
        feature_vector = extract_statistics(window)

        # Majority label in the window
        majority_label = window["Label"].mode().iloc[0]

        feature_vector["Label"] = majority_label

        engineered_rows.append(feature_vector)

    return pd.DataFrame(engineered_rows)

In [11]:
engineered_chunk = process_chunk(first_chunk)

print("Engineered Chunk Shape:")
print(engineered_chunk.shape)

engineered_chunk.head()

Engineered Chunk Shape:
(2496, 49)


,Flow Duration_Mean,Flow Duration_Std,Flow Duration_Variance,Flow Duration_Min,Flow Duration_Max,Flow Duration_Renyi,Flow Bytes/s_Mean,Flow Bytes/s_Std,Flow Bytes/s_Variance,Flow Bytes/s_Min,...,Total Backward Packets_Min,Total Backward Packets_Max,Total Backward Packets_Renyi,Average Packet Size_Mean,Average Packet Size_Std,Average Packet Size_Variance,Average Packet Size_Min,Average Packet Size_Max,Average Packet Size_Renyi,Label
0,10839530.69,2.843003e+07,8.082669e+14,1,119842965,0.595533,333457.721947,3.083312e+06,9.506815e+12,0.0,...,0,96,0.787368,119.622965,223.877889,50121.309327,0.0,1533.777778,1.270904,BENIGN
1,10505891.36,2.883378e+07,8.313870e+14,96,118514337,0.525330,42053.922346,2.446347e+05,5.984614e+10,0.0,...,0,201,0.696190,184.627261,317.521456,100819.875155,0.0,1784.444444,1.225666,BENIGN
2,12329517.29,3.187952e+07,1.016304e+15,97,118759341,0.586406,65741.641517,2.704786e+05,7.315865e+10,0.0,...,0,260,0.903468,252.496307,416.828398,173745.913406,0.0,1784.444444,1.502260,BENIGN
3,15855919.38,3.638650e+07,1.323977e+15,1,118759341,0.681770,376029.512487,3.089682e+06,9.546132e+12,0.0,...,0,260,1.081308,251.265778,418.410308,175067.185685,0.0,1784.444444,1.682233,BENIGN
4,15815469.62,3.650741e+07,1.332791e+15,1,118759341,0.647016,376991.772878,3.089569e+06,9.545437e+12,0.0,...,0,260,1.043943,254.374859,416.931181,173831.609710,0.0,1784.444444,1.531156,BENIGN


In [12]:
print(engineered_chunk.columns.tolist())

print("\n")

print(engineered_chunk["Label"].value_counts())

['Flow Duration_Mean', 'Flow Duration_Std', 'Flow Duration_Variance', 'Flow Duration_Min', 'Flow Duration_Max', 'Flow Duration_Renyi', 'Flow Bytes/s_Mean', 'Flow Bytes/s_Std', 'Flow Bytes/s_Variance', 'Flow Bytes/s_Min', 'Flow Bytes/s_Max', 'Flow Bytes/s_Renyi', 'Flow Packets/s_Mean', 'Flow Packets/s_Std', 'Flow Packets/s_Variance', 'Flow Packets/s_Min', 'Flow Packets/s_Max', 'Flow Packets/s_Renyi', 'Packet Length Mean_Mean', 'Packet Length Mean_Std', 'Packet Length Mean_Variance', 'Packet Length Mean_Min', 'Packet Length Mean_Max', 'Packet Length Mean_Renyi', 'Packet Length Std_Mean', 'Packet Length Std_Std', 'Packet Length Std_Variance', 'Packet Length Std_Min', 'Packet Length Std_Max', 'Packet Length Std_Renyi', 'Total Fwd Packets_Mean', 'Total Fwd Packets_Std', 'Total Fwd Packets_Variance', 'Total Fwd Packets_Min', 'Total Fwd Packets_Max', 'Total Fwd Packets_Renyi', 'Total Backward Packets_Mean', 'Total Backward Packets_Std', 'Total Backward Packets_Variance', 'Total Backward Packe

In [13]:
print("\nMissing Values:")
print(engineered_chunk.isnull().sum().sum())


Missing Values:
0


In [14]:
from pathlib import Path

# Remove previous output file if it exists
output_path = Path(OUTPUT_FILE)

if output_path.exists():
    output_path.unlink()
    print("Previous engineered dataset removed.")

total_chunks = 0
total_windows = 0

reader = pd.read_csv(
    INPUT_FILE,
    chunksize=CHUNK_SIZE
)

for chunk in tqdm(reader, desc="Processing Chunks"):

    engineered_chunk = process_chunk(chunk)

    total_windows += len(engineered_chunk)

    # Append to CSV
    engineered_chunk.to_csv(
        OUTPUT_FILE,
        mode="a",
        header=not output_path.exists(),
        index=False
    )

    total_chunks += 1

print("\nProcessing Complete!")
print(f"Chunks Processed : {total_chunks}")
print(f"Windows Created  : {total_windows}")
print(f"Saved To         : {OUTPUT_FILE}")

Processing Chunks: 1357it [2:31:29,  6.70s/it]


Processing Complete!
Chunks Processed : 1357
Windows Created  : 3385176
Saved To         : ../dataset/engineered/engineered_dataset.csv


In [15]:
engineered_df = pd.read_csv(OUTPUT_FILE)

print("Engineered Dataset Shape:")
print(engineered_df.shape)

print("\nFirst 5 Rows:")
engineered_df.head()

Engineered Dataset Shape:
(3385176, 49)

First 5 Rows:


,Flow Duration_Mean,Flow Duration_Std,Flow Duration_Variance,Flow Duration_Min,Flow Duration_Max,Flow Duration_Renyi,Flow Bytes/s_Mean,Flow Bytes/s_Std,Flow Bytes/s_Variance,Flow Bytes/s_Min,...,Total Backward Packets_Min,Total Backward Packets_Max,Total Backward Packets_Renyi,Average Packet Size_Mean,Average Packet Size_Std,Average Packet Size_Variance,Average Packet Size_Min,Average Packet Size_Max,Average Packet Size_Renyi,Label
0,10839530.69,2.843003e+07,8.082669e+14,1,119842965,0.595533,333457.721947,3.083312e+06,9.506815e+12,0.0,...,0,96,0.787368,119.622965,223.877889,50121.309327,0.0,1533.777778,1.270904,BENIGN
1,10505891.36,2.883378e+07,8.313870e+14,96,118514337,0.525330,42053.922346,2.446347e+05,5.984614e+10,0.0,...,0,201,0.696190,184.627261,317.521456,100819.875155,0.0,1784.444444,1.225666,BENIGN
2,12329517.29,3.187952e+07,1.016304e+15,97,118759341,0.586406,65741.641517,2.704786e+05,7.315865e+10,0.0,...,0,260,0.903468,252.496307,416.828398,173745.913406,0.0,1784.444444,1.502260,BENIGN
3,15855919.38,3.638650e+07,1.323977e+15,1,118759341,0.681770,376029.512487,3.089682e+06,9.546132e+12,0.0,...,0,260,1.081308,251.265778,418.410308,175067.185685,0.0,1784.444444,1.682233,BENIGN
4,15815469.62,3.650741e+07,1.332791e+15,1,118759341,0.647016,376991.772878,3.089569e+06,9.545437e+12,0.0,...,0,260,1.043943,254.374859,416.931181,173831.609710,0.0,1784.444444,1.531156,BENIGN


In [16]:
print("Dataset Shape:")
print(engineered_df.shape)

print("\n")

print("Missing Values:")
print(engineered_df.isnull().sum().sum())

print("\n")

print("Duplicate Rows:")
print(engineered_df.duplicated().sum())

print("\n")

print("Class Distribution:")
print(engineered_df["Label"].value_counts())

Dataset Shape:
(3385176, 49)


Missing Values:
0


Duplicate Rows:
731


Class Distribution:
Label
TFTP             974348
MSSQL            278897
Syn              274387
DrDoS_SNMP       257105
DrDoS_DNS        245125
DrDoS_MSSQL      219509
DrDoS_NetBIOS    197921
UDP              189301
NetBIOS          175580
DrDoS_UDP        154527
DrDoS_SSDP       128258
DrDoS_LDAP       106972
LDAP              93581
DrDoS_NTP         59660
UDP-lag           16500
Portmap            8835
BENIGN             4578
UDPLag               92
Name: count, dtype: int64


In [17]:
engineered_df.describe().T

,count,mean,std,min,25%,50%,75%,max
Flow Duration_Mean,3385176.0,1.202763e+06,3.973181e+06,1.000000,6.420000e+00,3.305900e+02,1.213575e+05,6.284720e+07
Flow Duration_Std,3385176.0,1.248363e+06,5.016995e+06,0.000000,1.508335e+01,2.908892e+03,4.554705e+05,5.815698e+07
Flow Duration_Variance,3385176.0,2.672864e+13,1.860095e+14,0.000000,2.275075e+02,8.461652e+06,2.074534e+11,3.382234e+15
Flow Duration_Min,3385176.0,2.849943e+04,5.014671e+05,1.000000,1.000000e+00,1.000000e+00,1.000000e+00,4.599455e+07
Flow Duration_Max,3385176.0,4.325129e+06,1.427313e+07,1.000000,5.100000e+01,2.162300e+04,3.031288e+06,1.200000e+08
Flow Duration_Renyi,3385176.0,5.659048e-01,4.921064e-01,-0.000000,1.765767e-01,4.231393e-01,8.013801e-01,3.791857e+00
Flow Bytes/s_Mean,3385176.0,7.670276e+08,7.553518e+08,0.000000,1.548006e+08,3.823093e+08,9.961200e+08,2.941920e+09
Flow Bytes/s_Std,3385176.0,4.138628e+08,2.825905e+08,0.000000,2.587983e+08,3.601941e+08,5.299231e+08,1.390822e+09
Flow Bytes/s_Variance,3385176.0,2.511398e+17,3.369202e+17,0.000000,6.697658e+16,1.297398e+17,2.808185e+17,1.934386e+18
Flow Bytes/s_Min,3385176.0,2.816607e+07,1.231527e+08,0.000000,6.591389e+02,1.243523e+05,1.679167e+07,2.896000e+09


In [19]:
print("Original Shape:", engineered_df.shape)

engineered_df = engineered_df.drop_duplicates().reset_index(drop=True)

print("New Shape:", engineered_df.shape)

print("Duplicates Remaining:", engineered_df.duplicated().sum())

Original Shape: (3385176, 49)
New Shape: (3384445, 49)
Duplicates Remaining: 0


In [20]:
engineered_df.to_csv(
    "../dataset/engineered/engineered_dataset.csv",
    index=False
)

engineered_df.to_pickle(
    "../dataset/engineered/engineered_dataset.pkl"
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
